In [0]:
dbutils.widgets.dropdown(name="environment", defaultValue="dev", choices=["dev", "qa", "prd"], label = "Select Environment")
env = dbutils.widgets.get("environment")
print(env)

catalog = f"saleslake_{env}"

bronzeTable = f"{catalog}.bronze_schema_{env}.rawDiscount"
silverTable = f"{catalog}.silver_schema_{env}.cleanedDiscount"
print(f"Source Table : {bronzeTable}")
print(f"Target Table : {silverTable}")

srcFileLoc=f"/Volumes/{catalog}/silver_{env}/vol_saleslake_src_files_{env}/manual_discount/"
# print(srcFileLoc)

In [0]:
spark.sql(f"""
INSERT INTO {silverTable}
SELECT DISTINCT
    CAST(TRIM(bronze.discount_id) AS INT)            AS discount_id,
    UPPER(TRIM(bronze.discount_code))                AS discount_code,
    TRIM(bronze.discount_name)                       AS discount_name,
    UPPER(TRIM(bronze.discount_type))                AS discount_type,
    CAST(TRIM(bronze.discount_value) AS DECIMAL(10,2)) AS discount_value,
    CAST(TRIM(bronze.min_amount) AS DECIMAL(18,2)) AS min_amount,
    CAST(TRIM(bronze.max_amount) AS DECIMAL(18,2)) AS max_amount,
    TO_DATE(TRIM(bronze.valid_from), 'yyyy-MM-dd')   AS valid_from,
    TO_DATE(TRIM(bronze.valid_to),   'yyyy-MM-dd')   AS valid_to,
    UPPER(TRIM(bronze.applicable_segment))           AS applicable_segment,
    UPPER(TRIM(bronze.applicable_category))          AS applicable_category,
    CAST(TRIM(bronze.usage_limit) AS INT) AS usage_limit,
    UPPER(TRIM(bronze.is_active))                    AS is_active,
    TO_DATE(TRIM(bronze.created_date), 'yyyy-MM-dd') AS created_date,
    CURRENT_TIMESTAMP() AS ingest_ts
FROM {bronzeTable} AS bronze
WHERE bronze.ingest_ts > (
    SELECT COALESCE(MAX(ingest_ts), TO_TIMESTAMP("1990-01-01"))
    FROM {silverTable}
)
ORDER BY discount_id
""")

print(f"Silver load complete for {silverTable}")

In [0]:
spark.sql(f"SELECT COUNT(*) AS row_count FROM {silverTable}").display()

In [0]:
# Discount Record Count
spark.sql(f"""
SELECT
    COUNT(*) AS total_discounts
FROM {silverTable}
""").show()

In [0]:
# Duplicate Discount Check
# Each discount_id should exist only once.

spark.sql(f"""
SELECT
    discount_id,
    COUNT(*) AS duplicate_count
FROM {silverTable}
GROUP BY discount_id
HAVING COUNT(*) > 1
""").show()

In [0]:
# Null Discount ID Check
spark.sql(f"""
SELECT
    COUNT(*) AS null_discount_id
FROM {silverTable}
WHERE discount_id IS NULL
""").show()

In [0]:
# Invalid Discount Amount Check
# Discount values and amount ranges should not be negative.

spark.sql(f"""
SELECT
    discount_id,
    discount_name,
    discount_value,
    min_amount,
    max_amount
FROM {silverTable}
WHERE discount_value < 0
   OR min_amount < 0
   OR max_amount < 0
""").show()

In [0]:
# Invalid Date Range Check
# valid_to should be greater than or equal to valid_from.

spark.sql(f"""
SELECT
    discount_id,
    discount_name,
    valid_from,
    valid_to
FROM {silverTable}
WHERE valid_to < valid_from
""").show()

In [0]:
# Active Flag Validation
# Only Y and N values should exist.

spark.sql(f"""
SELECT
    is_active,
    COUNT(*) AS record_count
FROM {silverTable}
GROUP BY is_active
ORDER BY record_count DESC
""").show()